In [3]:
import torch
import os
import gc
from diffusers import StableDiffusionXLPipeline
from diffusers.loaders import LoraLoaderMixin
from safetensors.torch import load_file

# 1. تنظيف الذاكرة
gc.collect()
torch.cuda.empty_cache()

# 2. المسارات
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, "..")) if "notebooks" in current_dir else current_dir

base_model_path = os.path.join(base_dir, "models", "base", "animagine-xl-3.1.safetensors")
lora_path = os.path.join(base_dir, "models", "lora", "my_anime_character_lora.safetensors")
output_fused_dir = os.path.join(base_dir, "models", "base", "animagine_fused_full")

print("... جاري تحميل النموذج الأساسي ...")
pipe = StableDiffusionXLPipeline.from_single_file(
    base_model_path,
    torch_dtype=torch.float16,
    use_safetensors=True
)

print("... جاري قراءة ملف الـ LoRA يدوياً ...")
if lora_path.endswith(".safetensors"):
    state_dict = load_file(lora_path)
else:
    state_dict = torch.load(lora_path, map_location="cpu")

print("... جاري تفكيك ودمج أوزان الـ UNet والـ Text Encoders ...")

def apply_lora_to_model(model, state_dict, prefix, lora_scale=1.0):
    """دالة لدمج أوزان الـ LoRA مباشرة في أوزان الموديل الأساسية بدون PEFT"""
    model_state = model.state_dict()
    updates = 0
    
    # البحث عن الأزواج lora_down و lora_up
    for key in list(state_dict.keys()):
        if prefix in key and "lora_down.weight" in key:
            up_key = key.replace("lora_down.weight", "lora_up.weight")
            alpha_key = key.replace("lora_down.weight", "alpha")
            
            if up_key in state_dict:
                down_weight = state_dict[key].to(dtype=torch.float32)
                up_weight = state_dict[up_key].to(dtype=torch.float32)
                
                # حساب المقياس (Alpha Scaling)
                rank = down_weight.shape[0]
                alpha = state_dict[alpha_key].item() if alpha_key in state_dict else rank
                scale = (alpha / rank) * lora_scale
                
                # حساب التغير في الأوزان W_delta = (Up @ Down) * scale
                if len(up_weight.shape) == 4: # Conv2d
                    delta = torch.mm(up_weight.flatten(1), down_weight.flatten(1)).view(up_weight.shape) * scale
                else: # Linear
                    delta = torch.matmul(up_weight, down_weight) * scale
                
                # إيجاد الاسم المطابق في الموديل الأصلي
                target_key = key.replace(prefix + "_", "").replace(".lora_down.weight", ".weight")
                target_key = target_key.replace(".", "_", target_key.count(".") - 1) if target_key not in model_state else target_key
                
                # مطابقة مفاتيح diffusers
                for m_key in model_state.keys():
                    clean_m_key = m_key.replace(".", "_")
                    clean_target_key = target_key.replace(".", "_")
                    if clean_m_key == clean_target_key or m_key.endswith(target_key):
                        model_state[m_key] += delta.to(device=model_state[m_key].device, dtype=model_state[m_key].dtype)
                        updates += 1
                        break
                        
    print(f"  - تم دمج {updates} طبقة لـ {prefix}")

# 1. دمج أوزان الـ UNet
apply_lora_to_model(pipe.unet, state_dict, prefix="lora_unet", lora_scale=1.0)

# 2. دمج أوزان الـ Text Encoder 1
if hasattr(pipe, "text_encoder") and pipe.text_encoder is not None:
    apply_lora_to_model(pipe.text_encoder, state_dict, prefix="lora_te1", lora_scale=1.0)
    apply_lora_to_model(pipe.text_encoder, state_dict, prefix="lora_te", lora_scale=1.0)

# 3. دمج أوزان الـ Text Encoder 2 (خاص بـ SDXL)
if hasattr(pipe, "text_encoder_2") and pipe.text_encoder_2 is not None:
    apply_lora_to_model(pipe.text_encoder_2, state_dict, prefix="lora_te2", lora_scale=1.0)

print(f"... جاري حفظ النموذج المدمج بالكامل في: {output_fused_dir} ...")
pipe.save_pretrained(output_fused_dir, safe_serialization=True)

# تنظيف الذاكرة
del pipe
gc.collect()
torch.cuda.empty_cache()

print("✅ تم دمج وحفظ النموذج الكامل (UNet + Text Encoders) بنجاح وبدون أخطاء!")

... جاري تحميل النموذج الأساسي ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


... جاري قراءة ملف الـ LoRA يدوياً ...
... جاري تفكيك ودمج أوزان الـ UNet والـ Text Encoders ...
  - تم دمج 0 طبقة لـ lora_unet
  - تم دمج 0 طبقة لـ lora_te1
  - تم دمج 0 طبقة لـ lora_te
  - تم دمج 192 طبقة لـ lora_te2
... جاري حفظ النموذج المدمج بالكامل في: c:\Users\OrbaTechs\Documents\Visual_Studio_18\cartoon-generator\models\base\animagine_fused_full ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ تم دمج وحفظ النموذج الكامل (UNet + Text Encoders) بنجاح وبدون أخطاء!


In [6]:
import torch
import gc
import os
from diffusers import AnimateDiffSDXLPipeline, MotionAdapter, AutoencoderKL
from diffusers.utils import export_to_gif
from diffusers.utils.logging import disable_progress_bar

disable_progress_bar()

# 1. تنظيف صارم للذاكرة قبل البدء
gc.collect()
torch.cuda.empty_cache()

# 2. المسارات
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, "..")) if "notebooks" in current_dir else current_dir

fused_model_dir = os.path.join(base_dir, "models", "base", "animagine_fused_full")

print("... جاري تحميل VAE المصلح لمنع البهتان ...")
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", 
    torch_dtype=torch.float32 # تشغيل الـ VAE بـ float32 لإلغاء اللون الرمادي تماماً
)

print("... جاري تحميل محرك الحركة Motion Adapter ...")
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-sdxl-beta", torch_dtype=torch.float16)

print("... جاري تحميل النموذج المدمج ...")
pipe = AnimateDiffSDXLPipeline.from_pretrained(
    fused_model_dir,
    motion_adapter=adapter,
    vae=vae,
    torch_dtype=torch.float16,
    use_safetensors=True
)

# 3. أقصى درجات توفير الذاكرة لكرت 8GB VRAM
pipe.enable_sequential_cpu_offload() # نقل أجزاء الموديل للـ RAM لتفادي الـ OOM
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

# 4. البرومبت مع التكثيف الألواني
prompt = "masterpiece, best quality, ultra-detailed, vibrant colors, highly saturated, sharp lighting, 1boy, mycustomboy, wearing green hooded adventurer cloak, dungeon background, torch flame flickering, hair swaying softly in wind, subtle head turn, looking at viewer"
negative_prompt = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, cropped, worst quality, low quality, blurry, static, washed out, low contrast, grayish, faded colors, pale"

print("... جاري توليد الأنيميشن بدقة 512x512 وبألوان مشبعة بدون OOM ...")

with torch.cuda.amp.autocast(dtype=torch.float16):
    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_frames=16,
        height=512, # الدقة الآمنة تماماً لـ 8GB VRAM
        width=512,
        guidance_scale=8.5, # رفع القيمة لإجبار الموديل على إظهار تفاصيل وألوان البرومبت
        num_inference_steps=25,
        generator=torch.Generator("cuda").manual_seed(42)
    )

frames = output.frames[0]

# 5. التصدير
output_gif_path = os.path.join(base_dir, "character_animation_fixed_vram.gif")
export_to_gif(frames, output_gif_path, fps=8)

print(f"🎉 تم التوليد بنجاح وبدون خروج للذاكرة! الملف محفوظ في: {output_gif_path}")

... جاري تحميل VAE المصلح لمنع البهتان ...
... جاري تحميل محرك الحركة Motion Adapter ...
... جاري تحميل النموذج المدمج ...


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

There are modules in UNetMotionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


... جاري توليد الأنيميشن بدقة 512x512 وبألوان مشبعة بدون OOM ...


C:\Users\OrbaTechs\AppData\Local\Temp\ipykernel_8500\3998621947.py:49: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


  0%|          | 0/25 [00:00<?, ?it/s]

🎉 تم التوليد بنجاح وبدون خروج للذاكرة! الملف محفوظ في: c:\Users\OrbaTechs\Documents\Visual_Studio_18\cartoon-generator\character_animation_fixed_vram.gif
